# nm_pets_3

1. Add the private dataset made by `python tools/package_code.py` (`dist/fpl-triage-code.zip`) as input.
2. Run all cells. Nothing needs editing.
3. Download `/kaggle/working/nm_pets_3.zip` at the end.

In [ ]:
import glob, json, os, re, shutil, statistics, subprocess, sys, threading, time

NAME = "nm_pets_3"
PLAN = {"oxford_pets": ["S_inst_gatefnm", "S_het_gatefnm", "N_sym25_gatefnm", "N_sym50_gatefnm", "N_sym75_gatefnm", "N_pair25_gatefnm", "N_pair50_gatefnm"]}
# runs in this notebook:
#   oxford_pets: S_inst_gatefnm, S_het_gatefnm, N_sym25_gatefnm, N_sym50_gatefnm, N_sym75_gatefnm, N_pair25_gatefnm, N_pair50_gatefnm
ARM_ARGS = {
    "S_inst_gatefnm": ["--attacker_model", "data", "--data_transform", "paper", "--attack_type", "none", "--malicious_client_frac", "0", "--noise_type", "instance", "--noise_rate", "0.4", "--noisy_client_frac", "0.3", "--sure", "true", "--sure_cal", "noise", "--sure_cal_flag", "false", "--sure_gate", "0.12", "--sure_cal_tmax", "1", "--sure_cal_floor", "true", "--sure_score", "xs", "--sure_tau", "0.072"],
    "S_het_gatefnm": ["--attacker_model", "data", "--data_transform", "paper", "--attack_type", "none", "--malicious_client_frac", "0", "--noise_type", "sym", "--noise_mode", "hetero", "--noise_rate", "0.8", "--noisy_client_frac", "0.5", "--sure", "true", "--sure_cal", "noise", "--sure_cal_flag", "false", "--sure_gate", "0.12", "--sure_cal_tmax", "1", "--sure_cal_floor", "true", "--sure_score", "xs", "--sure_tau", "0.072"],
    "N_sym25_gatefnm": ["--attacker_model", "data", "--data_transform", "paper", "--attack_type", "none", "--malicious_client_frac", "0", "--noise_type", "sym", "--noise_rate", "0.25", "--noisy_client_frac", "1.0", "--sure", "true", "--sure_cal", "noise", "--sure_cal_flag", "false", "--sure_gate", "0.12", "--sure_cal_tmax", "1", "--sure_cal_floor", "true", "--sure_score", "xs", "--sure_tau", "0.072"],
    "N_sym50_gatefnm": ["--attacker_model", "data", "--data_transform", "paper", "--attack_type", "none", "--malicious_client_frac", "0", "--noise_type", "sym", "--noise_rate", "0.5", "--noisy_client_frac", "1.0", "--sure", "true", "--sure_cal", "noise", "--sure_cal_flag", "false", "--sure_gate", "0.12", "--sure_cal_tmax", "1", "--sure_cal_floor", "true", "--sure_score", "xs", "--sure_tau", "0.072"],
    "N_sym75_gatefnm": ["--attacker_model", "data", "--data_transform", "paper", "--attack_type", "none", "--malicious_client_frac", "0", "--noise_type", "sym", "--noise_rate", "0.75", "--noisy_client_frac", "1.0", "--sure", "true", "--sure_cal", "noise", "--sure_cal_flag", "false", "--sure_gate", "0.12", "--sure_cal_tmax", "1", "--sure_cal_floor", "true", "--sure_score", "xs", "--sure_tau", "0.072"],
    "N_pair25_gatefnm": ["--attacker_model", "data", "--data_transform", "paper", "--attack_type", "none", "--malicious_client_frac", "0", "--noise_type", "pair", "--noise_rate", "0.25", "--noisy_client_frac", "1.0", "--sure", "true", "--sure_cal", "noise", "--sure_cal_flag", "false", "--sure_gate", "0.12", "--sure_cal_tmax", "1", "--sure_cal_floor", "true", "--sure_score", "xs", "--sure_tau", "0.072"],
    "N_pair50_gatefnm": ["--attacker_model", "data", "--data_transform", "paper", "--attack_type", "none", "--malicious_client_frac", "0", "--noise_type", "pair", "--noise_rate", "0.5", "--noisy_client_frac", "1.0", "--sure", "true", "--sure_cal", "noise", "--sure_cal_flag", "false", "--sure_gate", "0.12", "--sure_cal_tmax", "1", "--sure_cal_floor", "true", "--sure_score", "xs", "--sure_tau", "0.072"]
}
TIMES, ROUNDS = 5, 50
BUDGET_HOURS = 10.6          # a run that would end past this is skipped, not started
HARD_HOURS = 10.75          # a run still going at this point is killed, so the zip is always made
SEC_PER_ROUND = {"oxford_pets": 31.0, "dtd": 37.5, "caltech101": 103.0, "mnist": 10.5, "eurosat": 10.0}
MEASURED = {"caltech101|pflcn16": 29.4, "caltech101|pflcn16D": 141.0, "caltech101|gatefcn16": 104.0, "eurosat|pflcn16": 6.3, "eurosat|pflcn16D": 11.0, "mnist|pflcn16": 10.0, "mnist|pflcn16D": 17.2, "oxford_pets|pflcn16": 13.1, "oxford_pets|pflcn16D": 40.6, "dtd|pflcn16": 13.1, "oxford_pets|gatefcn32": 53.1, "oxford_pets|pflcn32": 17.1, "dtd|gatefcn32": 67.1, "dtd|pflcn32": 19.1, "eurosat|gatefcn32": 8.8, "eurosat|pflcn32": 7.5, "mnist|gatefcn32": 12.6, "mnist|pflcn32": 10.3, "caltech101|pflcn32": 32.8, "caltech101|pflcn32L": 55.2, "oxford_pets|pflcn32L": 28.8, "dtd|pflcn32L": 32.1, "eurosat|pflcn32L": 8.9, "mnist|pflcn32L": 12.3, "oxford_pets|gatefnm": 19.8, "dtd|gatefnm": 22.4, "caltech101|gatefnm": 59.0, "mnist|gatefnm": 10.9, "eurosat|gatefnm": 6.4}

WORK = "/kaggle/working"
OUT = f"{WORK}/{NAME}"
DATA_ROOT = "/tmp/fpl_data"
CLIP_CACHE = f"{WORK}/clip_cache"
CODE = f"{WORK}/code"
NOTEBOOK_START = time.time()
hits = glob.glob("/kaggle/input/**/flip/main.py", recursive=True)
assert hits, "attach the dataset built by tools/package_code.py"
CODE_INPUT = os.path.dirname(os.path.dirname(hits[0]))
print("code:", CODE_INPUT)

In [ ]:
subprocess.run("nvidia-smi", shell=True)
if not os.path.exists(CODE):
    shutil.copytree(CODE_INPUT, CODE)
os.makedirs(f"{OUT}/logs", exist_ok=True)

# Kaggle unpacks .gz files inside datasets, which removes CLIP's BPE vocab: put it back
import gzip, hashlib, urllib.request
BPE = f"{CODE}/flip/flcore/models/clip/bpe_simple_vocab_16e6.txt.gz"
if not os.path.isfile(BPE):
    if os.path.isfile(BPE[:-3]):
        with open(BPE[:-3], "rb") as fi, gzip.open(BPE, "wb") as fo:
            shutil.copyfileobj(fi, fo)
    else:
        urllib.request.urlretrieve("https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz", BPE)
        assert hashlib.sha256(open(BPE, "rb").read()).hexdigest() == \
            "924691ac288e54409236115652ad4aa250f48203de50a9e4722a6ecd48d6804a"

need = {"ftfy": "ftfy", "regex": "regex", "gdown": "gdown", "h5py": "h5py",
        "cv2": "opencv-python-headless", "termcolor": "termcolor",
        "tensorboard": "tensorboard", "sklearn": "scikit-learn", "pyarrow": "pyarrow"}
missing = []
for mod, pkg in need.items():
    try:
        __import__(mod)
    except ImportError:
        missing.append(pkg)
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
import gdown

SPLITS = {"caltech101": ("1hyarUivQE36mY6jSomru6Fjd-JzwcCzN", "split_zhou_Caltech101.json"),
          "dtd": ("1u3_QfB467jqHgNXC00UIzbLZRQCg2S7x", "split_zhou_DescribableTextures.json"),
          "oxford_pets": ("1501r8Ber4nNKvmlFVQZ8SeUHTcdTTEqs", "split_zhou_OxfordPets.json")}


def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)


def fetch(urls, dst):
    # the Oxford VGG host sometimes answers 500; thor.robots.ox.ac.uk serves the same files
    for url in urls:
        rc = subprocess.run(f"wget -q --timeout=60 --tries=2 -O {dst} {url}", shell=True).returncode
        if rc == 0 and os.path.getsize(dst) > 1_000_000:
            print("got", url)
            return
        print("failed", url)
    raise RuntimeError(f"every mirror failed: {urls}")


# Hugging Face mirrors with the original file names and JPEG bytes, used when both Oxford hosts are
# down (checked 2026-10-31: banded_0002.jpg has the same md5 as the server's Oxford copy)
HF_MIRRORS = {
    "dtd": ("split_zhou_DescribableTextures.json", "https://huggingface.co/datasets/jxie/dtd/resolve/main/data/",
            ["train-00000-of-00001-ebe06d6bffa9b3c3.parquet",
             "validation-00000-of-00001-4c0e2e1fd7fbd259.parquet",
             "test-00000-of-00001-b49cbb6beb34aad1.parquet"]),
    "oxford_pets": ("split_zhou_OxfordPets.json",
                    "https://huggingface.co/datasets/timm/oxford-iiit-pet/resolve/main/data/",
                    ["train-00000-of-00001.parquet", "test-00000-of-00001.parquet"]),
}


def from_hf(name, d):
    # every image goes where Zhou's split file expects it (images/<rel>); all of them must be there
    import pyarrow.parquet as pq
    split_file, base, parts = HF_MIRRORS[name]
    split = json.load(open(f"{CODE}/splits/{split_file}"))
    where = {p.split("/")[-1]: p for part in split.values() for p, _, _ in part}
    written = set()
    for part in parts:
        sh(f"wget -q --timeout=60 --tries=3 -O /tmp/mirror.parquet {base}{part}")
        for img in pq.read_table("/tmp/mirror.parquet", columns=["image"]).column("image").to_pylist():
            rel = where.get(os.path.basename(img["path"] or ""))
            if rel:
                os.makedirs(os.path.dirname(f"{d}/images/{rel}"), exist_ok=True)
                with open(f"{d}/images/{rel}", "wb") as f:
                    f.write(img["bytes"])
                written.add(rel)
        os.remove("/tmp/mirror.parquet")
    lost = set(where.values()) - written
    assert not lost, f"{len(lost)} {name} images not in the mirror, e.g. {sorted(lost)[:3]}"
    print(name, "from the Hugging Face mirror:", len(written), "images")


def prepare(name):
    d = f"{DATA_ROOT}/{name}"
    os.makedirs(d, exist_ok=True)
    if name == "caltech101" and not os.path.isdir(f"{d}/101_ObjectCategories"):
        sh("wget -q -O /tmp/c.zip https://data.caltech.edu/records/mzrjq-6wc02/files/caltech-101.zip")
        sh("rm -rf /tmp/cz && unzip -q /tmp/c.zip -d /tmp/cz")
        tgz = glob.glob("/tmp/cz/**/101_ObjectCategories.tar.gz", recursive=True)[0]
        sh(f"tar -xzf {tgz} -C {d} && rm -rf /tmp/c.zip /tmp/cz")
    if name == "dtd" and not os.path.isdir(f"{d}/images"):
        try:
            fetch(["https://thor.robots.ox.ac.uk/dtd/dtd-r1.0.1.tar.gz",
                   "https://www.robots.ox.ac.uk/~vgg/data/dtd/download/dtd-r1.0.1.tar.gz"], "/tmp/dtd.tar.gz")
            sh(f"tar -xzf /tmp/dtd.tar.gz -C {DATA_ROOT} && rm /tmp/dtd.tar.gz")
        except RuntimeError:
            from_hf("dtd", d)
    if name == "oxford_pets" and not os.path.isdir(f"{d}/images"):
        try:
            for part in ("images", "annotations"):
                fetch([f"https://thor.robots.ox.ac.uk/pets/{part}.tar.gz",
                       f"https://www.robots.ox.ac.uk/~vgg/data/pets/data/{part}.tar.gz"], f"/tmp/{part}.tar.gz")
                sh(f"tar -xzf /tmp/{part}.tar.gz -C {d} && rm /tmp/{part}.tar.gz")
        except RuntimeError:
            from_hf("oxford_pets", d)     # only images/ and the split file are read
    if name in ("mnist", "eurosat"):
        # no CoOp split file: MNIST is written out from torchvision, EuroSAT comes from Zenodo, and
        # both get their train/val/test folders here with seed 0, as server/prep_mnist_eurosat.py does
        sys.path[:0] = [f"{CODE}/flip", f"{CODE}/server"]
        from prep_mnist_eurosat import EuroSAT, MNIST, get_eurosat
        if name == "mnist":
            MNIST(d, 0).preprocess()
        else:
            get_eurosat(d)
            EuroSAT(d, 0).preprocess()
        print(name, "ready")
        return
    file_id, fname = SPLITS[name]
    if not os.path.exists(f"{d}/{fname}"):
        if os.path.isfile(f"{CODE}/splits/{fname}"):      # shipped with the code
            shutil.copy(f"{CODE}/splits/{fname}", d)
        else:
            gdown.download(id=file_id, output=f"{d}/{fname}", quiet=True)
    print(name, "ready")


for ds in PLAN:
    prepare(ds)

In [ ]:
COMMON = ["--benchmark", "global", "--image_backbone", "RN50", "--optim_name", "sgd",
          "--lr_scheduler", "cos", "--split_alpha", "0.1", "--loss_type", "ce", "--central", "false",
          "--num_clients", "10", "--num_shot", "8", "--optim_momentum", "0.9",
          "--local_learning_rate", "0.002", "--batch_size", "16", "--eval_scaler", "2",
          "--num_prompt", "1", "--local_epochs", "1", "--prompt_batch_size", "2", "--precision", "amp",
          "--log_metrics", "true",    # macro P/R/F1 + round cost -> metrics_seed_<seed>.json
          "--num_workers", "4", "--data_root", DATA_ROOT, "--verbose2", "--cache_test_feats", "true"]
INDEX = f"{OUT}/run_index.json"
ROUND_RE = re.compile(r"round (\d+), .*?top1: ([\d.]+)%, elapsed: ([\d.]+)s")
rates = {tuple(k.split("|")): v for k, v in MEASURED.items()}


def run(ds, arm):
    index = json.load(open(INDEX)) if os.path.exists(INDEX) else {}
    tag = f"{ds}_{arm}"
    if index.get(tag, {}).get("returncode") == 0:
        return
    used = (time.time() - NOTEBOOK_START) / 3600
    seeds = TIMES                # an arm may override the seed count with its own --times
    for i, a in enumerate(ARM_ARGS[arm][:-1]):
        if a == "--times":
            seeds = int(ARM_ARGS[arm][i + 1])
    # rate per (dataset, method). A method not measured yet uses the dataset's FedCLEAR rate, x 1.6
    # for a run with --diag_scores (measured 1.07-1.6x FedCLEAR at 16 shots; PromptFL without
    # scores is faster than FedCLEAR on every measured dataset). The old "1.5x the slowest known
    # method" rule gave FedCLEAR Caltech 44 s per round from the PromptFL rate (real 94-104).
    # "L": a PromptFL S_ run logs --log_signals (zero-shot signals every round), 1.68x slower than the
    # same method without it (Caltech pflcn32 S 3.91 h vs N 2.32 h, server 2026-11-11)
    method = arm.rsplit("_", 1)[-1] + ("D" if "--diag_scores" in ARM_ARGS[arm] else "")
    if "--log_signals" in ARM_ARGS[arm] and "--diag_scores" not in ARM_ARGS[arm] and "--sure" not in ARM_ARGS[arm]:
        method += "L"
    rate = rates.get((ds, method), SEC_PER_ROUND[ds] * (1.6 if method.endswith("D") else 1.0))
    need = rate * (ROUNDS + 1) * seeds / 3600 + 0.1
    if used + need > BUDGET_HOURS:
        print(f"SKIPPED {tag}: needs ~{need:.1f} h, {BUDGET_HOURS - used:.1f} h left")
        return
    print(f"start {tag} (~{need:.1f} h, notebook at {used:.1f} h)", flush=True)
    run_dir, log = f"{OUT}/runs/{tag}", f"{OUT}/logs/{tag}.txt"
    os.makedirs(run_dir, exist_ok=True)
    env = dict(os.environ, PYTHONPATH=f"{CODE}/flip", FLIP_CLIP_CACHE=CLIP_CACHE, PYTHONUNBUFFERED="1")
    cmd = [sys.executable, f"{CODE}/flip/main.py", "--dataset", ds, "--prompt_algo", "CoOp",
           "--times", str(TIMES), "--global_rounds", str(ROUNDS), *COMMON, *ARM_ARGS[arm]]
    t0, seen = time.time(), []
    with open(log, "w") as f:
        proc = subprocess.Popen(cmd, cwd=run_dir, env=env, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        left = HARD_HOURS * 3600 - (time.time() - NOTEBOOK_START)
        killer = threading.Timer(max(left, 0), proc.kill)
        killer.start()
        for line in proc.stdout:
            f.write(line)
            m = ROUND_RE.search(line)
            if m:
                seen.append(float(m.group(3)))
                if int(m.group(1)) % 10 == 0:
                    print(f"  [{tag}] round {m.group(1)} top1 {m.group(2)}% "
                          f"({(time.time() - t0) / 60:.0f} min)", flush=True)
            elif "Traceback" in line or "Error" in line:
                print(f"  [{tag}] {line.strip()}", flush=True)
        proc.wait()
        killer.cancel()
    if proc.returncode != 0 and time.time() - NOTEBOOK_START >= HARD_HOURS * 3600:
        print(f"KILLED {tag} at the hard limit of {HARD_HOURS} h", flush=True)
    if seen:
        rates[(ds, method)] = statistics.fmean(seen)
    index[tag] = {"dataset": ds, "arm": arm, "args": list(ARM_ARGS[arm]),
                  "returncode": proc.returncode, "seconds": round(time.time() - t0, 1)}
    json.dump(index, open(INDEX, "w"), indent=1)
    print(f"{tag}: rc={proc.returncode} in {(time.time() - t0) / 3600:.2f} h")
    if proc.returncode != 0:
        print("".join(open(log).readlines()[-30:]))


for ds, arms in PLAN.items():
    for arm in arms:
        run(ds, arm)

In [ ]:
# last10 accuracy per run, then the zip to download
for tag in sorted(json.load(open(INDEX)) if os.path.exists(INDEX) else {}):
    seeds, cur = [], None
    for line in open(f"{OUT}/logs/{tag}.txt", errors="replace"):
        m = ROUND_RE.search(line)
        if m:
            if int(m.group(1)) == 0:
                cur = []
                seeds.append(cur)
            cur.append(float(m.group(2)))
    vals = [statistics.fmean(s[-10:]) for s in seeds if len(s) > 10]
    if vals:
        print(f"{tag}: {statistics.fmean(vals):.2f} over {len(vals)} seeds {[round(v, 2) for v in vals]}")
archive = shutil.make_archive(f"{WORK}/{NAME}", "zip", OUT)
print(archive, "%.1f MB" % (os.path.getsize(archive) / 1e6))